# PII Anonymization Leakage

Hashing an email address and calling the data "anonymized" is one of the most common
compliance mistakes in data engineering. When quasi-identifiers like ZIP code, date of
birth, and gender remain in cleartext, an attacker can cross-reference public datasets
and re-identify individuals — even without reversing the hash.

This notebook demonstrates the attack, measures k-anonymity, and shows how to
generalize quasi-identifiers until the data is genuinely safe.

In [1]:
!pip install faker pandas dataprof==0.7.0 --quiet

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

## Generate Synthetic User Data

We create 5 000 fake users with fields that look like a typical CRM export:
name, email, date of birth, ZIP code, gender, and a transaction amount.

ZIP codes are drawn from a small pool (~50) to simulate realistic low-cardinality
geographic data.

In [2]:
import pandas as pd
import numpy as np
import hashlib
from faker import Faker

fake = Faker("en_US")
Faker.seed(42)
np.random.seed(42)

N = 5_000

# Pool of ~50 ZIP codes to create realistic low cardinality
zip_pool = [fake.zipcode() for _ in range(50)]

users = pd.DataFrame({
    "name": [fake.name() for _ in range(N)],
    "email": [fake.unique.email() for _ in range(N)],
    "date_of_birth": [fake.date_of_birth(minimum_age=18, maximum_age=75) for _ in range(N)],
    "zip_code": np.random.choice(zip_pool, size=N),
    "gender": np.random.choice(["M", "F", "NB"], size=N, p=[0.48, 0.48, 0.04]),
    "transaction_amount": np.round(np.random.uniform(5, 2000, size=N), 2),
})

print(f"Users: {len(users)} rows")
display(users.head())

Users: 5000 rows


,name,email,date_of_birth,zip_code,gender,transaction_amount
0,Renee Blair,holdendominic@example.net,1960-10-26,55893,M,216.72
1,Valerie Gray,wnovak@example.net,1962-11-11,92425,F,1378.85
2,Sandra Montgomery,grantamanda@example.net,1971-05-28,77898,F,1957.41
3,Laura Bush,joe99@example.net,1952-02-18,28722,F,1454.58
4,Darren Roberts,krista02@example.org,1966-04-09,18790,F,1857.82


## Anti-Pattern: Hash the Email and Call It a Day

The naive approach: apply SHA-256 to the email (the only "direct identifier") and
leave everything else untouched. This feels secure — the hash is irreversible. But
the combination of quasi-identifiers (ZIP + DOB + gender) can uniquely identify
individuals.

Research by Latanya Sweeney showed that **87% of the US population** can be uniquely
identified by just {ZIP, DOB, gender}.

In [3]:
# "Anonymize" by hashing emails
naive_anon = users.copy()
naive_anon["email"] = naive_anon["email"].apply(
    lambda x: hashlib.sha256(x.encode()).hexdigest()
)
naive_anon = naive_anon.drop(columns=["name"])  # drop name too, feeling safe

print("Naively 'anonymized' data:")
display(naive_anon.head())

Naively 'anonymized' data:


,email,date_of_birth,zip_code,gender,transaction_amount
0,fbb0a657294c16ec54c605bf5055bd202522bc1eea7f4f...,1960-10-26,55893,M,216.72
1,6e264f02fedc798dcf34ed2442b853d02cf997a8198452...,1962-11-11,92425,F,1378.85
2,103194e373b1526e8734be3f60fde8f299294995b87333...,1971-05-28,77898,F,1957.41
3,5fcb4eadf7574fddcc99418f9d50dccd37f3b3062ab0c6...,1952-02-18,28722,F,1454.58
4,be342841e2435ede53d3ed19c1b73de6e33172369b3569...,1966-04-09,18790,F,1857.82


In [4]:
# Measure k-anonymity on quasi-identifiers
quasi_ids = ["zip_code", "date_of_birth", "gender"]
group_sizes = naive_anon.groupby(quasi_ids).size()

k_min = group_sizes.min()
unique_count = (group_sizes == 1).sum()
total_groups = len(group_sizes)

print(f"k-anonymity (minimum group size): k = {k_min}")
print(f"Groups with k=1 (uniquely identifiable): {unique_count} / {total_groups} "
      f"({unique_count / total_groups * 100:.1f}%)")
print(f"\nIndividuals in k=1 groups: {unique_count} people can be re-identified")
print("\nk=1 means this person's quasi-identifier combo is unique in the dataset.")
print("An attacker with a voter registry can link the hash back to a real name.")

# Show the distribution
print("\nGroup size distribution:")
print(group_sizes.describe())

k-anonymity (minimum group size): k = 1
Groups with k=1 (uniquely identifiable): 4986 / 4993 (99.9%)

Individuals in k=1 groups: 4986 people can be re-identified

k=1 means this person's quasi-identifier combo is unique in the dataset.
An attacker with a voter registry can link the hash back to a real name.

Group size distribution:
count    4993.000000
mean        1.001402
std         0.037420
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         2.000000
dtype: float64


## Profiling with dataprof: Spotting Dangerous Cardinality

Even without manually checking k-anonymity, automated profiling can raise red flags.
Columns with high cardinality relative to the dataset size (like `date_of_birth`
with ~20,000 possible values over 5,000 rows) are inherently risky as
quasi-identifiers.

In [5]:
import dataprof as dp

# Profile the non-hashed columns — we're looking for dangerous cardinality
profile = dp.profile(naive_anon.drop(columns=["email"]))

print(f"Rows: {profile.rows}  |  Quality: {profile.quality_score:.1f}%")
print()

# Extract the key quasi-identifier risk signals column by column
for col in profile:
    cp = profile[col]
    uniqueness_pct = cp.uniqueness_ratio * 100
    risk = "HIGH RISK" if uniqueness_pct > 50 else ("MEDIUM" if uniqueness_pct > 10 else "low")
    print(f"{col:<22} type={cp.data_type:<8} unique={cp.unique_count:>5} ({uniqueness_pct:.1f}%)  [{risk}]")
    if cp.patterns:
        for p in cp.patterns:
            print(f"  ⚠ pattern detected: {p.name} ({p.match_percentage:.0f}% of values)")

print()
print("date_of_birth has", profile["date_of_birth"].unique_count,
      "distinct values across", profile.rows, "rows —",
      f"{profile['date_of_birth'].unique_count / profile.rows * 100:.0f}% cardinality.")
print("Combined with zip_code and gender, this is a quasi-identifier fingerprint.")

Rows: 5000  |  Quality: 83.8%

zip_code               type=integer  unique=   50 (1.0%)  [low]
  ⚠ pattern detected: CAP (IT) (100% of values)
  ⚠ pattern detected: ZIP Code (US) (100% of values)
transaction_amount     type=float    unique= 1000 (20.0%)  [MEDIUM]
date_of_birth          type=date     unique= 1000 (20.0%)  [MEDIUM]
gender                 type=string   unique=    3 (0.1%)  [low]

date_of_birth has 1000 distinct values across 5000 rows — 20% cardinality.
Combined with zip_code and gender, this is a quasi-identifier fingerprint.


## Dogfooding Notes — dataprof 0.7.0

Observations from profiling this dataset:

- **`zip_code` detected as `integer` with `CAP (IT)` pattern at 100%**: known false positive — the Italian postal code regex `^\d{5}$` matches any 5-digit US ZIP. Numeric IDs and ZIPs will always trigger this. Don't rely on pattern names alone to identify geographic fields; use column name heuristics alongside.
- **`date_of_birth` correctly inferred as `date`** with 1,000 unique values across 5,000 rows (20% cardinality) — this is exactly the signal we want to surface. High-cardinality date columns are quasi-identifier red flags.
- **`gender` has 3 unique values** (M/F/NB) — low cardinality alone isn't dangerous, but as a combination factor with DOB and ZIP it dramatically increases re-identification risk. dataprof surfaces uniqueness per-column; cross-column combinatorial risk requires manual k-anonymity analysis.
- **Suppression rate 97.8%**: the synthetic dataset uses only 50 ZIP codes with exact DOB, so generalizing to 3-digit ZIP prefix leaves very few groups with k ≥ 5. In real CRM data with hundreds of ZIP codes, suppression would be far less severe.

## Correct Approach: k-Anonymity via Generalization

To achieve genuine anonymization, we must **reduce the precision** of quasi-identifiers
until every combination matches at least `k` individuals:

1. **date_of_birth** → generalize to **birth_year** (reduces ~20K values to ~57).
2. **zip_code** → generalize to **3-digit prefix** (reduces ~50 to ~40).
3. **Suppress** any remaining groups with fewer than `k` members.

A common threshold is **k ≥ 5**: any individual is indistinguishable from at least 4 others.

In [6]:
K_THRESHOLD = 5

safe_anon = naive_anon.copy()

# Step 1: Generalize date_of_birth to birth_year
safe_anon["birth_year"] = pd.to_datetime(safe_anon["date_of_birth"]).dt.year
safe_anon = safe_anon.drop(columns=["date_of_birth"])

# Step 2: Generalize zip_code to 3-digit prefix
safe_anon["zip_3"] = safe_anon["zip_code"].astype(str).str[:3]
safe_anon = safe_anon.drop(columns=["zip_code"])

# Step 3: Suppress groups with k < threshold
generalized_qi = ["zip_3", "birth_year", "gender"]
group_sizes_new = safe_anon.groupby(generalized_qi).transform("size")
mask_safe = group_sizes_new >= K_THRESHOLD

n_suppressed = (~mask_safe).sum()
safe_anon = safe_anon[mask_safe].reset_index(drop=True)

print(f"Suppressed {n_suppressed} rows in groups with k < {K_THRESHOLD}")
print(f"Remaining rows: {len(safe_anon)}")

# Verify new k-anonymity
k_new = safe_anon.groupby(generalized_qi).size().min()
print(f"\nNew k-anonymity: k = {k_new} (target was k >= {K_THRESHOLD})")

print("\nSafe anonymized data:")
display(safe_anon.head(10))

Suppressed 4890 rows in groups with k < 5
Remaining rows: 110

New k-anonymity: k = 5 (target was k >= 5)

Safe anonymized data:


,email,gender,transaction_amount,birth_year,zip_3
0,d87fb148cc0a57e9ff0700e80c9f2f6c9baeb9f70b94b3...,M,759.66,1966,369
1,001d9bbf62f741df9a99af62721d39741d82a362d242b4...,F,1824.19,1999,667
2,f82aee80f83e0dd8406a8dc74c0320a7679d0888ac2e68...,F,1529.25,1960,719
3,49a5aee0093639a9a6667ce6fc63c5b49d4ebaf6fa261f...,M,1331.91,1982,369
4,6a9a44e1fc6fc37718496ed7cc8a168ec9b2c132f9d0de...,F,579.96,1960,777
5,8a262b4d0193cec71b7e676ca1139d2dfc50ffec684508...,F,978.21,2005,127
6,0b1b7e5ce278bf8a931e730f9d497f31eec4c2888764bb...,F,591.45,1966,369
7,0938fdb84e0672bed78b63ef018d982ed66b1c99eebab7...,F,1759.40,1999,667
8,9b486627912fbe3c62c2daa0bcb6abb34cdcf2a13eeb80...,M,1263.12,2005,150
9,1a4d71ee614aa8e632894454b23dc48994e7d9eda17936...,F,1309.26,1964,719


In [7]:
# Compare before/after
print("=== Before (naive hashing) ===")
qi_before = naive_anon.groupby(["zip_code", "date_of_birth", "gender"]).size()
print(f"  k-anonymity:       k = {qi_before.min()}")
print(f"  Unique individuals: {(qi_before == 1).sum()}")
print(f"  Total rows:        {len(naive_anon)}")

print(f"\n=== After (generalization + suppression, k >= {K_THRESHOLD}) ===")
qi_after = safe_anon.groupby(generalized_qi).size()
print(f"  k-anonymity:       k = {qi_after.min()}")
print(f"  Unique individuals: {(qi_after == 1).sum()}")
print(f"  Total rows:        {len(safe_anon)}")
print(f"  Data loss:         {n_suppressed} rows ({n_suppressed / len(naive_anon) * 100:.1f}%)")

=== Before (naive hashing) ===
  k-anonymity:       k = 1
  Unique individuals: 4986
  Total rows:        5000

=== After (generalization + suppression, k >= 5) ===
  k-anonymity:       k = 5
  Unique individuals: 0
  Total rows:        110
  Data loss:         4890 rows (97.8%)


## Takeaways

| Technique | Protects Against | Risk If Skipped |
| :--- | :--- | :--- |
| Hash direct identifiers (email, name) | Direct lookup attacks | Identity exposed in cleartext |
| Generalize quasi-identifiers (DOB → year, ZIP → prefix) | Linkage attacks via cross-referencing | 87% of US population uniquely identifiable by {ZIP, DOB, gender} |
| Suppress small groups (k < threshold) | Residual re-identification in rare combos | Outlier individuals remain unique |
| Profile with dataprof / audit cardinality | Overlooked high-cardinality columns | Hidden quasi-identifiers slip through |

### Legal context

Under GDPR, **pseudonymization** (e.g., hashing emails) still counts as personal data
processing and requires a legal basis. Only **anonymization** — where re-identification is
"reasonably impossible" — falls outside GDPR scope. k-anonymity with sufficient generalization
is one recognized approach, alongside l-diversity and t-closeness for sensitive attributes.

**Bottom line:** hashing alone is pseudonymization, not anonymization. If quasi-identifiers
remain, you still have personal data in the eyes of the law.